
# Indian Premier League (IPL) Match & Player Performance Analytics

## 1. Problem Statement
The goal of this project is to build a reliable and robust ETL (Extract, Transform, Load) pipeline that ingests raw IPL match and ball-by-ball delivery datasets, structures them inside a relational PostgreSQL database, and processes them to compute advanced sports analytics. Specifically, we aim to design and implement a custom **Dream11 Fantasy Points Calculator** for batters. This involves calculating historical match-by-match metrics (such as runs, balls faced, boundaries, and strike rates) and mapping them to dynamic, multi-tier fantasy scoring rules—producing clean, export-ready datasets for downstream analytic engines.
```

In [ ]:
!pip install -q pandas sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 55.1 MB/s eta 0:00:00



### Step 1: Library Installation & Environment Setup
We begin by installing the necessary database connectors and object-relational mapping (ORM) dependencies: `sqlalchemy` for handling database connectivity and SQL translation, and `psycopg2-binary`/`psycopg` for connecting to the PostgreSQL database server.


In [23]:
!pip install -q "psycopg[binary]"


### Step 2: Database Ingestion & Schema Normalization
In this step, we read the raw CSV files for both matches and individual ball deliveries. We sanitize and normalize the columns (converting them to lowercase, stripping whitespaces, and replacing special characters with underscores) before executing bulk transactional loads into PostgreSQL using chunking for memory optimization.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine,text
from google.colab import userdata
engine=create_engine(userdata.get("database_url"))
files={
    "matches.csv":"matches",
    "deliveries.csv":"deliveries"

}
for file,table in files.items():
  df=pd.read_csv(file)
  df.columns=(df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True))
  print(f"{file}: {df.shape[0]} rows, {df.shape[1]} columns")
  df.to_sql(table,engine,if_exists='replace',index=False,chunksize=3000,method='multi')
  with engine.connect() as conn:
    n=conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
  assert n == len(df), f"Row count mismatch in {table}"
  print(f"{table}: {n} rows loaded")




matches.csv: 1095 rows, 20 columns
matches: 1095 rows loaded
deliveries.csv: 260920 rows, 17 columns
deliveries: 260920 rows loaded



### Step 3: Loading Normalized Data into pandas DataFrames
Here, we retrieve the fully-loaded tables from PostgreSQL into memory-efficient pandas DataFrames to prepare for complex feature engineering and aggregations.


In [24]:
import pandas as pd
from sqlalchemy import create_engine
from google.colab import userdata
engine=create_engine(userdata.get("database_url"))
matches_df=pd.read_sql("SELECT * FROM matches",engine)
deliveries_df=pd.read_sql("select * from deliveries",engine)


### Step 4: Exploratory Inspection
We preview the structured `matches_df` and `deliveries_df` structures to ensure data types, columns, and records are properly aligned.


In [ ]:
matches_df

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,None,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,None,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,None,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,None,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,None,BF Bowden,K Hariharan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,1426307,2024,Hyderabad,2024-05-19,League,Abhishek Sharma,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Punjab Kings,Sunrisers Hyderabad,Punjab Kings,bat,Sunrisers Hyderabad,wickets,4.0,215.0,20.0,N,None,Nitin Menon,VK Sharma
1091,1426309,2024,Ahmedabad,2024-05-21,Qualifier 1,MA Starc,"Narendra Modi Stadium, Ahmedabad",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,8.0,160.0,20.0,N,None,AK Chaudhary,R Pandit
1092,1426310,2024,Ahmedabad,2024-05-22,Eliminator,R Ashwin,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,wickets,4.0,173.0,20.0,N,None,KN Ananthapadmanabhan,MV Saidharshan Kumar
1093,1426311,2024,Chennai,2024-05-24,Qualifier 2,Shahbaz Ahmed,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals,field,Sunrisers Hyderabad,runs,36.0,176.0,20.0,N,None,Nitin Menon,VK Sharma


In [ ]:
deliveries_df

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,None,None,None
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,None,None,None
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260915,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,5,SS Iyer,AK Markram,VR Iyer,1,0,1,None,0,None,None,None
260916,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,6,VR Iyer,AK Markram,SS Iyer,1,0,1,None,0,None,None,None
260917,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,1,VR Iyer,Shahbaz Ahmed,SS Iyer,1,0,1,None,0,None,None,None
260918,1426312,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,2,SS Iyer,Shahbaz Ahmed,VR Iyer,1,0,1,None,0,None,None,None



### Step 5: Feature Engineering & Aggregation of Match-by-Match Metrics
To compute individual batter statistics for each unique match, we group ball-by-ball data to extract:
- **Runs:** Total runs scored by the batter.
- **Balls Faced:** Total balls faced by the batter.
- **Fours (4s):** Total boundaries scoring exactly 4 runs.
- **Sixes (6s):** Total boundary hits scoring exactly 6 runs.

We then merge these temporary structures, fill missing values (representing zeroes for batters who hit no boundaries), and calculate the corresponding Strike Rate (SR).


In [ ]:
# Initialize the DataFrame first if you want to create a new one
df = pd.DataFrame()
df=deliveries_df[['match_id','batter']].drop_duplicates()
df
df



,match_id,batter
0,335982,SC Ganguly
1,335982,BB McCullum
34,335982,RT Ponting
82,335982,DJ Hussey
108,335982,Mohammad Hafeez
...,...,...
260849,1426312,B Kumar
260853,1426312,Rahmanullah Gurbaz
260860,1426312,SP Narine
260862,1426312,VR Iyer


In [ ]:
runs=deliveries_df.groupby(['match_id','batter'])['total_runs'].sum().reset_index()
runs

balls=deliveries_df.groupby(['match_id','batter'])['total_runs'].count().reset_index()
balls
fours=deliveries_df.query("batsman_runs==4").groupby(['match_id','batter'])['total_runs'].count().reset_index()
sixes=deliveries_df.query("batsman_runs==6").groupby(['match_id','batter'])['total_runs'].count().reset_index()
final=runs.merge(balls,on=['match_id','batter'],how='left',suffixes=('_runs','_balls')).merge(fours,on=['match_id','batter'],how='left').merge(sixes,on=['match_id','batter'],how='left')

In [ ]:
final.columns

Index(['match_id', 'batter', 'total_runs_runs', 'total_runs_balls',
       'total_runs_x', 'total_runs_y'],
      dtype='object')

In [ ]:
final.fillna(0,inplace=True)
final

,match_id,batter,total_runs_runs,total_runs_balls,total_runs_x,total_runs_y
0,335982,AA Noffke,11,12,1.0,0.0
1,335982,B Akhil,0,2,0.0,0.0
2,335982,BB McCullum,169,77,10.0,13.0
3,335982,CL White,6,10,0.0,0.0
4,335982,DJ Hussey,12,12,1.0,0.0
...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0
16511,1426312,SS Iyer,6,3,1.0,0.0
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0
16513,1426312,TM Head,0,1,0.0,0.0


In [ ]:
final.rename(columns={
    'total_runs_runs':'runs',
    'total_runs_balls':'balls',
    'total_runs_x':'fours',
    'total_runs_y':'sixes'
})

,match_id,batter,runs,balls,fours,sixes
0,335982,AA Noffke,11,12,1.0,0.0
1,335982,B Akhil,0,2,0.0,0.0
2,335982,BB McCullum,169,77,10.0,13.0
3,335982,CL White,6,10,0.0,0.0
4,335982,DJ Hussey,12,12,1.0,0.0
...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0
16511,1426312,SS Iyer,6,3,1.0,0.0
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0
16513,1426312,TM Head,0,1,0.0,0.0


In [ ]:
final = final.rename(columns={
    'total_runs_runs': 'runs',
    'total_runs_balls': 'balls',
    'total_runs_x': 'fours',
    'total_runs_y': 'sixes'
})

final['sr'] = round(((final['runs'] / final['balls']) * 100), 2)
final

,match_id,batter,runs,balls,fours,sixes,sr
0,335982,AA Noffke,11,12,1.0,0.0,91.67
1,335982,B Akhil,0,2,0.0,0.0,0.00
2,335982,BB McCullum,169,77,10.0,13.0,219.48
3,335982,CL White,6,10,0.0,0.0,60.00
4,335982,DJ Hussey,12,12,1.0,0.0,100.00
...,...,...,...,...,...,...,...
16510,1426312,SP Narine,6,2,0.0,1.0,300.00
16511,1426312,SS Iyer,6,3,1.0,0.0,200.00
16512,1426312,Shahbaz Ahmed,8,7,0.0,1.0,114.29
16513,1426312,TM Head,0,1,0.0,0.0,0.00



### Step 6: Implementing the Dream11 Fantasy Point Scoring Logic
We write a customized conditional rules function `dream11()` to grade performance based on standard fantasy sports benchmarks, incorporating:
- Milestone bonuses (e.g., half-centuries, centuries)
- Deductions (e.g., ducks)
- Strike rate bonuses/penalties for players meeting a minimum threshold of 10 balls faced.
```

In [25]:
def dream11(row):
  score=0
  score=row['runs']+row['fours']+2*row['sixes']
  if row['runs']>=50 or row['runs']<100:
    score=score+8
  elif row['runs']>=100:
    score=score+16
  elif row['runs']==0:
    score=score-2
  if row['balls']>=10:
    if row['sr']>170:
      score=score+6
    elif row['sr']>150 and row['sr']<=170:
      score=score+4
    elif row['sr']>130 and row['sr']<=150:
      score=score+2
    elif row['sr']>60 and row['sr']<=70:
      score=score-2
    elif row['sr']>50 and row['sr']<=60:
      score=score-4
    elif row['sr']<=50:
      score=score-6
    else:
      pass

  return score

final.apply(dream11,axis=1)
export=final[['match_id','batter','score']]
export

,match_id,batter,score
0,335982,AA Noffke,20.0
1,335982,B Akhil,8.0
2,335982,BB McCullum,219.0
3,335982,CL White,10.0
4,335982,DJ Hussey,21.0
...,...,...,...
16510,1426312,SP Narine,16.0
16511,1426312,SS Iyer,15.0
16512,1426312,Shahbaz Ahmed,18.0
16513,1426312,TM Head,8.0



### Step 7: Exporting Final Fantasy Scores to the Relational Database
Finally, we persist our calculated metrics back into a new PostgreSQL table (`dream11_df`) to make this engineered dataset readily available for visualization tools or predictive modeling pipelines.
```

In [26]:
export.to_sql(
    name='dream11_df'
    ,con=engine
    ,if_exists='replace'
    ,index=False
    ,chunksize=3000
    ,method='multi'
)

-6

In [27]:
pd.read_sql("select * from dream11_df",engine)

,match_id,batter,score
0,335982,AA Noffke,20.0
1,335982,B Akhil,8.0
2,335982,BB McCullum,219.0
3,335982,CL White,10.0
4,335982,DJ Hussey,21.0
...,...,...,...
16510,1426312,SP Narine,16.0
16511,1426312,SS Iyer,15.0
16512,1426312,Shahbaz Ahmed,18.0
16513,1426312,TM Head,8.0
